# TN4 — Test cuối trên GHIJ, DS-TCN 64 kênh tầm nhìn 61

## Đây là số công bố

Train trên **đủ 8 người ABCDEFKL**, chấm **một lần** trên **GHIJ** — 537 buổi
ghi của 4 người chưa từng xuất hiện ở bất kỳ bước chọn cấu hình nào.

Khác với `run_cv.py`: ở đó model chỉ thấy 6 người mỗi fold và điểm dùng để
**chọn**. Ở đây model thấy cả 8 người và điểm dùng để **báo cáo**. Theo đúng
`docs/PROTOCOL.md` mục 6.

## Cấu hình — đã chốt xong, không chọn gì thêm ở đây

| | |
|---|---|
| model | `ds_tcn --channels 64 --kernel_size 3 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element` |
| tham số | **37.081** |
| tầm nhìn | 61 |
| loss | `mse_pearson --alpha 0,6` |
| điểm dev của cấu hình này | **0,780028** (TN3, 4 fold, 1 seed) |
| cùng kiến trúc, MSE thuần | 0,760878 |

`alpha 0,6` chọn từ **TN3 trên tập dev**, là mức cao nhất của chính bề
rộng kênh này. GHIJ không tham gia vào lựa chọn đó.

**Một giới hạn phải ghi khi báo cáo.** TN3 cho thấy thứ hạng giữa các alpha
không chuyển được giữa hai bề rộng kênh — tốp 3 của c64 (0,6 · 0,5 · 0,4) và
của c192 (0,2 · 0,3 · 0,0) không giao nhau mức nào, và biên độ dao động bên
trong mỗi cột cùng cỡ với dao động giữa các seed. Nên `alpha 0,6` là
**mức tốt nhất đo được trên dev**, không phải mức tối ưu đã chứng minh.

Điều TN3 chứng minh được là **20/20 mức alpha đều hơn MSE thuần**, ở cả hai cấu
hình. Đó mới là phát biểu đem vào luận văn.

## Ba seed

Ba seed cho ra `seed_std` của chính con số công bố — thứ trả lời được câu hỏi
"nhỡ ăn may thì sao". Mỗi lần chạy khoảng **25 phút**, tổng khoảng
**1,3 giờ**.

`run_final_test.py` **tự nén và chép sang Drive sau mỗi lần chạy**, tên tệp nén
có cả cấu hình lẫn seed nên không đè nhau. Không cần ô lưu riêng.

Dừng giữa chừng cũng được: mở lại, chạy ô khôi phục ở mục 1 rồi bấm tiếp seed
còn thiếu. Seed đã xong sẽ in `đã có kết quả macro ... — không chạy lại.`

## 1. Chuẩn bị Colab

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Tải mã nguồn.

In [ ]:
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

Lấy `by_user/` và `windows/` từ Drive. Test cuối đọc `windows/final_train/`, cắt gộp cả 8 người theo đúng thứ tự MobiVital.

In [ ]:
!python scripts/restore_processed_data_on_drive.py

Khôi phục các seed đã chạy.

**Chạy ô này mỗi khi mở lại notebook.** Nó gộp `runs/*/summary.csv` vào `runs/summary.csv` — chỗ `run_final_test.py` tra để biết seed nào đã xong.

In [ ]:
# Khôi phục kết quả đã chạy trước khi phiên bị ngắt.
#
# Mỗi tệp nén chứa một bản runs/<thực nghiệm>/summary.csv của riêng nó. Giải
# hết vào cùng một thư mục runs/ thì tệp giải sau ĐÈ summary.csv của tệp trước.
# Mà sorted() xếp "..._a0_corr0.9..." đứng SAU "..._a0.9_corr0.9...", vì trong
# bảng mã ký tự "_" lớn hơn "." — nên bản ít dòng nhất lại là bản đè cuối cùng.
# Sửa: mỗi tệp nén giải vào một thư mục tạm riêng, gộp mọi dòng lại rồi mới ghi
# runs/summary.csv một lần. Thứ tự giải nén không còn ảnh hưởng gì nữa.
import csv, glob, os, shutil, subprocess, tempfile

MAU_ZIP = "/content/drive/MyDrive/mobivital/tn4_*c64_*.zip"

rows, seen = [], set()

def collect(summary_path):
    for r in csv.DictReader(open(summary_path)):
        key = (r.get("experiment"), r.get("run_id"))
        if key not in seen:
            seen.add(key)
            rows.append(r)

for f in sorted(glob.glob(MAU_ZIP)):
    tmp = tempfile.mkdtemp()
    subprocess.run(["unzip", "-oq", f, "-d", tmp], check=True)
    for s in glob.glob(tmp + "/*/summary.csv"):
        collect(s)
        os.remove(s)   # gộp xong thì bỏ, để bước chép dưới không đè nhau nữa
    # Checkpoint và curve.csv nằm trong thư mục riêng của từng lần chạy, tên
    # không trùng nhau, nên chép chồng lên runs/ là an toàn.
    for d in os.listdir(tmp):
        shutil.copytree(tmp + "/" + d, "runs/" + d, dirs_exist_ok=True)
    shutil.rmtree(tmp)

# Dòng đã sinh ra trong chính phiên này cũng phải giữ lại.
if os.path.exists("runs/summary.csv"):
    collect("runs/summary.csv")

if rows:
    # run_cv.py tra runs/summary.csv, còn tệp nén chỉ có runs/<thực nghiệm>/summary.csv
    cols = []
    for r in rows:
        for k in r:
            if k not in cols:
                cols.append(k)
    with open("runs/summary.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols, restval="")
        w.writeheader()
        w.writerows(rows)
print("khôi phục", len(rows), "dòng vào runs/summary.csv")

## 2. Kiểm bản cài đặt

Số tham số phải ra đúng **37.081**.

In [ ]:
!python scripts/check_model.py --model ds_tcn --channels 64 \
    --kernel_size 3 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

## 3. Ba seed

Mỗi ô train lại từ đầu trên ABCDEFKL rồi chấm GHIJ.

**seed 0**

In [ ]:
!python scripts/run_final_test.py --experiment tn4 --model ds_tcn --channels 64 \
    --kernel_size 3 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.6 --seed 0

**seed 1**

In [ ]:
!python scripts/run_final_test.py --experiment tn4 --model ds_tcn --channels 64 \
    --kernel_size 3 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.6 --seed 1

**seed 2**

In [ ]:
!python scripts/run_final_test.py --experiment tn4 --model ds_tcn --channels 64 \
    --kernel_size 3 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.6 --seed 2

## 4. Kết quả

`score_macro` là **Pearson macro theo người** trên GHIJ — trung bình từng người, rồi trung bình bốn người.

In [ ]:
import csv, os
import statistics as st

# run_final_test.py ghi mỗi lần chạy MỘT dòng, không có dòng TONG như run_cv.py.
rows = []
if os.path.exists("runs/tn4/summary.csv"):
    rows = [r for r in csv.DictReader(open("runs/tn4/summary.csv"))
            if "_c64_" in r["run_id"]]

diem = {}
for r in rows:
    diem[int(r["seed"])] = float(r["score_macro"])

for s in sorted(diem):
    print("  seed", s, " ", round(diem[s], 6))
if len(diem) >= 2:
    v = list(diem.values())
    print("  " + "-" * 34)
    print("  trung bình", round(st.mean(v), 6))
    print("  seed_std  ", round(st.stdev(v), 6))
print()
print("  MỐC ĐỐI CHIẾU trên GHIJ")
for ten, d in (("MobiVital công bố", "0,819"),
               ("LSTM-352, 1.502.713 tham số", "0,810302"),
               ("LSTM-67", "0,801683"),
               ("DS-TCN-64 bản đầu", "0,795783")):
    print("   ", ten.ljust(30), d)

## 5. Ngắt phiên

In [ ]:
from google.colab import runtime
runtime.unassign()